# 07 - Integrated Research Analysis

This notebook connects the earlier project components instead of evaluating them in isolation. It asks:

1. When do anomaly detection and failure classification agree or disagree?
2. Which sensor patterns characterize false positives and false negatives as groups?
3. Do group-level SHAP explanations support the individual cases studied in Notebook 06?
4. Are the base and engineered model results stable across five folds?
5. Are SHAP explanations faithful, and how stable are selected LIME explanations?
6. How does the preferred model change when missed failures and false alarms have different costs?

This is a synthesis and validation stage. It does not introduce a more complex predictive model.

## 1. Setup

The notebook reuses the same dataset split and classifier configuration as Notebooks 03–06. The anomaly preprocessing intentionally matches Notebook 03 exactly so its published results remain reproducible.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from IPython.display import display
from lime.lime_tabular import LimeTabularExplainer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


def find_project_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src" / "predictive_maintenance").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from predictive_maintenance.classification import FailureClassificationExperiment
from predictive_maintenance.config import INPUT_FEATURES, NUMERIC_FEATURES, RANDOM_STATE
from predictive_maintenance.data_loader import load_ai4i_dataset
from predictive_maintenance.explainability import get_transformed_train_test

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "integrated_research_analysis"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

print(f"Project root: {PROJECT_ROOT}")

## 2. Train the Base Classifier

The classifier uses the same six safe inputs, depth-4 decision tree, balanced class weights, random seed, and 0.5 threshold as Notebook 04.

In [ ]:
data = load_ai4i_dataset()

classification_experiment = FailureClassificationExperiment(
    max_depth=4,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
classification_experiment.fit(data)

classification_results = (
    classification_experiment.predictions_frame(threshold=0.5)
    .copy()
)
classification_results.insert(0, "dataset_row_index", classification_results.index)

display(classification_experiment.metrics_frame(threshold=0.5))

## 3. Reproduce the Anomaly Detector

Notebook 03 one-hot encoded `Type` with the first category dropped, standardized the five sensors, trained Isolation Forest only on normal training records, and used contamination 0.05. Repeating those choices is necessary for consistent same-record comparison.

In [ ]:
split_data = classification_experiment.split_data

x_train_encoded = pd.get_dummies(
    split_data.x_train,
    columns=["Type"],
    drop_first=True,
    dtype=int,
)
x_test_encoded = pd.get_dummies(
    split_data.x_test,
    columns=["Type"],
    drop_first=True,
    dtype=int,
)
x_test_encoded = x_test_encoded.reindex(columns=x_train_encoded.columns, fill_value=0)

scaler = StandardScaler()
x_train_anomaly = x_train_encoded.copy()
x_test_anomaly = x_test_encoded.copy()
x_train_anomaly[NUMERIC_FEATURES] = scaler.fit_transform(
    x_train_encoded[NUMERIC_FEATURES]
)
x_test_anomaly[NUMERIC_FEATURES] = scaler.transform(
    x_test_encoded[NUMERIC_FEATURES]
)

normal_train_mask = split_data.y_train == 0
anomaly_model = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
anomaly_model.fit(x_train_anomaly.loc[normal_train_mask])

raw_anomaly_predictions = anomaly_model.predict(x_test_anomaly)
anomaly_flags = (raw_anomaly_predictions == -1).astype(int)
anomaly_scores = anomaly_model.decision_function(x_test_anomaly)

print(f"Anomalies flagged: {anomaly_flags.sum()} of {len(anomaly_flags)}")
print("Failure/anomaly overlap matrix:")
print(confusion_matrix(split_data.y_test, anomaly_flags))

The executed result reproduced 113 anomaly flags, including 16 of the 68 test failures. This check confirms that the integrated analysis did not silently change the earlier experiment.

## 4. Combine Both Signals on the Same Records

The two binary outputs form four transparent signal groups:

- **Neither:** no anomaly and no classified failure
- **Classifier only:** predicted failure but not anomalous
- **Anomaly only:** unusual behavior but predicted no failure
- **Both:** anomalous and predicted failure

These are descriptive decision-support signals, not calibrated risk categories.

In [ ]:
combined = classification_results.copy()
combined["anomaly_flag"] = anomaly_flags
combined["anomaly_score"] = anomaly_scores

combined["signal_group"] = np.select(
    [
        (combined["anomaly_flag"] == 0)
        & (combined["predicted_machine_failure"] == 0),
        (combined["anomaly_flag"] == 0)
        & (combined["predicted_machine_failure"] == 1),
        (combined["anomaly_flag"] == 1)
        & (combined["predicted_machine_failure"] == 0),
        (combined["anomaly_flag"] == 1)
        & (combined["predicted_machine_failure"] == 1),
    ],
    ["neither", "classifier_only", "anomaly_only", "both"],
    default="unknown",
)

signal_order = ["neither", "classifier_only", "anomaly_only", "both"]
combined["signal_group"] = pd.Categorical(
    combined["signal_group"], categories=signal_order, ordered=True
)

signal_summary = (
    combined.groupby("signal_group", observed=True)
    .agg(
        records=("actual_machine_failure", "size"),
        actual_failures=("actual_machine_failure", "sum"),
        mean_failure_probability=("predicted_failure_probability", "mean"),
        mean_anomaly_score=("anomaly_score", "mean"),
    )
    .reset_index()
)
signal_summary["observed_failure_rate"] = (
    signal_summary["actual_failures"] / signal_summary["records"]
)

display(signal_summary)
signal_summary.to_csv(OUTPUT_DIR / "signal_group_summary.csv", index=False)
combined.to_csv(OUTPUT_DIR / "combined_record_results.csv", index=False)

### Signal Interpretation

The executed result is:

| Signal group | Records | Failures | Observed failure rate |
|---|---:|---:|---:|
| Neither | 1,716 | 4 | 0.23% |
| Classifier only | 171 | 48 | 28.07% |
| Anomaly only | 78 | 1 | 1.28% |
| Both | 35 | 15 | 42.86% |

Agreement between both methods produces the highest observed failure rate. The classifier-only group contains most failures, showing that many known failure patterns are not globally unusual. The anomaly-only group mostly represents unusual non-failure behavior. Four failures receive neither signal, so absence of both flags does not guarantee safety.

In [ ]:
count_matrix = pd.crosstab(
    combined["anomaly_flag"],
    combined["predicted_machine_failure"],
).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
count_matrix.index = ["Anomaly = 0", "Anomaly = 1"]
count_matrix.columns = ["Classifier = 0", "Classifier = 1"]

failure_count_matrix = pd.crosstab(
    combined["anomaly_flag"],
    combined["predicted_machine_failure"],
    values=combined["actual_machine_failure"],
    aggfunc="sum",
).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
failure_count_matrix.index = count_matrix.index
failure_count_matrix.columns = count_matrix.columns
failure_rate_matrix = failure_count_matrix.div(count_matrix).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(count_matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Records by anomaly/classifier agreement")
sns.heatmap(
    failure_rate_matrix,
    annot=True,
    fmt=".2%",
    cmap="YlOrRd",
    vmin=0,
    vmax=0.5,
    ax=axes[1],
)
axes[1].set_title("Observed failure rate by signal group")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "signal_agreement_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Group-Level Classification Error Analysis

Notebook 06 explained one TP, FP, FN, and TN. Here we test whether those local observations are representative of their complete error groups.

In [ ]:
actual = combined["actual_machine_failure"]
predicted = combined["predicted_machine_failure"]
combined["error_type"] = np.select(
    [
        (actual == 1) & (predicted == 1),
        (actual == 0) & (predicted == 1),
        (actual == 1) & (predicted == 0),
        (actual == 0) & (predicted == 0),
    ],
    ["true_positive", "false_positive", "false_negative", "true_negative"],
    default="unknown",
)

error_order = ["true_positive", "false_positive", "false_negative", "true_negative"]
combined["error_type"] = pd.Categorical(
    combined["error_type"], categories=error_order, ordered=True
)

error_group_summary = (
    combined.groupby("error_type", observed=True)[
        NUMERIC_FEATURES + ["predicted_failure_probability"]
    ]
    .mean()
    .round(3)
)

display(error_group_summary)
error_group_summary.to_csv(OUTPUT_DIR / "error_group_sensor_means.csv")

### Error-Group Finding

False negatives average **1,678.4 rpm, 30.56 Nm, and 197 minutes of tool wear**. Compared with true positives, they operate at higher speed, lower torque, and greater wear. This supports the local false-negative explanation from Notebook 06 and suggests a repeated blind spot rather than an isolated example.

False positives average 161.59 minutes of wear and 44.97 Nm torque, both above the true-negative means. Their average failure probability is 0.853, showing that many false alarms are confident rather than borderline.

In [ ]:
plot_features = ["Rotational speed", "Torque", "Tool wear"]
long_error_means = (
    error_group_summary[plot_features]
    .reset_index()
    .melt(id_vars="error_type", var_name="feature", value_name="mean_value")
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feature in zip(axes, plot_features):
    subset = long_error_means[long_error_means["feature"] == feature]
    sns.barplot(data=subset, x="error_type", y="mean_value", color="#287271", ax=ax)
    ax.set_title(feature)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
fig.suptitle("Mean sensor values by classification outcome")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "error_group_sensor_means.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Group-Level SHAP Analysis

Mean absolute SHAP values measure which features most strongly influence each outcome group. Signed mean SHAP values show whether the average contribution pushes toward failure or no failure.

In [ ]:
transformed = get_transformed_train_test(
    classification_experiment.model,
    classification_experiment.split_data,
)
tree_classifier = classification_experiment.model.named_steps["classifier"]
shap_explainer = shap.TreeExplainer(tree_classifier)
raw_shap_values = shap_explainer.shap_values(transformed["x_test"])

if isinstance(raw_shap_values, list):
    failure_shap_values = np.asarray(raw_shap_values[1])
elif np.asarray(raw_shap_values).ndim == 3:
    failure_shap_values = np.asarray(raw_shap_values)[:, :, 1]
else:
    failure_shap_values = np.asarray(raw_shap_values)

clean_feature_names = [
    name.replace("numeric__", "").replace("categorical__", "")
    for name in transformed["feature_names"]
]
shap_frame = pd.DataFrame(failure_shap_values, columns=clean_feature_names)
shap_frame["error_type"] = combined["error_type"].astype(str).values

group_absolute_shap = (
    shap_frame.groupby("error_type")[clean_feature_names]
    .apply(lambda group: group.abs().mean())
    .reindex(error_order)
)
group_signed_shap = (
    shap_frame.groupby("error_type")[clean_feature_names]
    .mean()
    .reindex(error_order)
)

display(group_absolute_shap.round(4))
group_absolute_shap.to_csv(OUTPUT_DIR / "group_mean_absolute_shap.csv")
group_signed_shap.to_csv(OUTPUT_DIR / "group_mean_signed_shap.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(group_absolute_shap.T, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Mean absolute SHAP by outcome")
axes[0].set_xlabel("")
axes[0].set_ylabel("")
sns.heatmap(
    group_signed_shap.T,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    ax=axes[1],
)
axes[1].set_title("Mean signed SHAP by outcome")
axes[1].set_xlabel("")
axes[1].set_ylabel("")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "group_shap_heatmaps.png", dpi=200, bbox_inches="tight")
plt.show()

The false-negative row is especially important: negative signed contributions from the dominant operating features reveal which learned rules suppress failure warnings across the complete missed-failure group. This is stronger evidence than one local explanation alone.

## 7. SHAP Faithfulness Check

For Tree SHAP, the baseline plus all feature contributions should reconstruct the model's failure output. We measure the reconstruction error across all 2,000 test records.

In [ ]:
expected_values = np.asarray(shap_explainer.expected_value)
shap_baseline = float(expected_values[1] if expected_values.ndim else expected_values)
model_probabilities = classification_experiment.predict_proba()
reconstructed_probabilities = shap_baseline + failure_shap_values.sum(axis=1)
reconstruction_error = np.abs(model_probabilities - reconstructed_probabilities)

shap_faithfulness = pd.DataFrame(
    {
        "metric": [
            "shap_baseline",
            "mean_absolute_reconstruction_error",
            "maximum_absolute_reconstruction_error",
        ],
        "value": [
            shap_baseline,
            reconstruction_error.mean(),
            reconstruction_error.max(),
        ],
    }
)

display(shap_faithfulness)
shap_faithfulness.to_csv(OUTPUT_DIR / "shap_faithfulness.csv", index=False)

The executed check produced a mean absolute reconstruction error of $3.11\times10^{-14}$ and a maximum error of $3.24\times10^{-14}$. This numerical-zero result demonstrates fidelity to the fitted tree output. It does not establish physical correctness or causal validity.

## 8. LIME Stability Check

LIME is stochastic because it samples a local neighborhood. We repeat explanations for the first false positive and false negative across five seeds. Stability is summarized using local surrogate $R^2$ and top-three-feature Jaccard overlap relative to seed 0.

The transformed one-hot type columns are treated numerically to match Notebook 06; this remains a documented limitation.

In [ ]:
def feature_from_condition(condition, feature_names):
    matches = [name for name in feature_names if name in condition]
    return max(matches, key=len) if matches else condition


lime_case_positions = {
    "false_positive": int(np.flatnonzero((actual == 0) & (predicted == 1))[0]),
    "false_negative": int(np.flatnonzero((actual == 1) & (predicted == 0))[0]),
}

lime_run_rows = []
top_feature_sets = {}
for case_name, row_position in lime_case_positions.items():
    for seed in range(5):
        lime_explainer = LimeTabularExplainer(
            training_data=transformed["x_train"],
            feature_names=transformed["feature_names"],
            class_names=["No failure", "Failure"],
            mode="classification",
            random_state=seed,
        )
        explanation = lime_explainer.explain_instance(
            transformed["x_test"][row_position],
            tree_classifier.predict_proba,
            labels=[1],
            num_features=8,
            num_samples=3000,
        )
        weighted_conditions = sorted(
            explanation.as_list(label=1), key=lambda item: abs(item[1]), reverse=True
        )
        top_three = {
            feature_from_condition(condition, transformed["feature_names"])
            for condition, _ in weighted_conditions[:3]
        }
        top_feature_sets[(case_name, seed)] = top_three

        for rank, (condition, weight) in enumerate(weighted_conditions, start=1):
            lime_run_rows.append(
                {
                    "case_type": case_name,
                    "seed": seed,
                    "surrogate_r2": explanation.score,
                    "rank": rank,
                    "feature": feature_from_condition(
                        condition, transformed["feature_names"]
                    ),
                    "condition": condition,
                    "weight_for_failure": weight,
                }
            )

lime_stability_runs = pd.DataFrame(lime_run_rows)
stability_rows = []
for case_name in lime_case_positions:
    reference = top_feature_sets[(case_name, 0)]
    jaccards = []
    for seed in range(5):
        current = top_feature_sets[(case_name, seed)]
        jaccards.append(len(reference & current) / len(reference | current))
    case_runs = lime_stability_runs[lime_stability_runs["case_type"] == case_name]
    seed_scores = case_runs.groupby("seed")["surrogate_r2"].first()
    stability_rows.append(
        {
            "case_type": case_name,
            "mean_surrogate_r2": seed_scores.mean(),
            "std_surrogate_r2": seed_scores.std(),
            "mean_top3_jaccard_vs_seed0": np.mean(jaccards),
            "minimum_top3_jaccard_vs_seed0": np.min(jaccards),
        }
    )

lime_stability_summary = pd.DataFrame(stability_rows)
display(lime_stability_summary)
lime_stability_runs.to_csv(OUTPUT_DIR / "lime_stability_runs.csv", index=False)
lime_stability_summary.to_csv(OUTPUT_DIR / "lime_stability_summary.csv", index=False)

The false-positive explanation had perfect top-three overlap across the five runs, but its mean surrogate $R^2$ was only 0.1115. The false-negative explanation was less stable: its mean top-three Jaccard overlap was 0.70, its minimum overlap was 0.50, and its mean surrogate $R^2$ was 0.0970. The feature rankings therefore have some qualitative consistency, but the low $R^2$ values warn that these local linear surrogates approximate the tree poorly. LIME should be presented as supporting evidence, not as an exact account of the model.

## 9. Five-Fold Validation of Engineered Features

The earlier base-versus-engineered result came from one split. Stratified five-fold cross-validation tests whether the direction of the result is stable across multiple held-out partitions.

In [ ]:
def add_engineered_features(frame):
    engineered = frame.copy()
    engineered["Temperature difference"] = (
        engineered["Process temperature"] - engineered["Air temperature"]
    )
    engineered["Power proxy"] = engineered["Torque"] * engineered["Rotational speed"]
    engineered["Wear torque interaction"] = (
        engineered["Tool wear"] * engineered["Torque"]
    )
    engineered["Speed torque ratio"] = (
        engineered["Rotational speed"] / (engineered["Torque"] + 1e-6)
    )
    return engineered


engineered_numeric_features = NUMERIC_FEATURES + [
    "Temperature difference",
    "Power proxy",
    "Wear torque interaction",
    "Speed torque ratio",
]


def build_tree_pipeline(numeric_features, class_weight="balanced"):
    preprocessor = ColumnTransformer(
        [
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), ["Type"]),
        ]
    )
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "classifier",
                DecisionTreeClassifier(
                    max_depth=4,
                    class_weight=class_weight,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


x_base = data[INPUT_FEATURES].copy()
x_engineered = add_engineered_features(x_base)
y = data["Machine failure"].copy()

cross_validation = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=RANDOM_STATE
)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

cv_rows = []
for model_name, x_values, numeric_features in [
    ("base", x_base, NUMERIC_FEATURES),
    ("engineered", x_engineered, engineered_numeric_features),
]:
    scores = cross_validate(
        build_tree_pipeline(numeric_features),
        x_values,
        y,
        cv=cross_validation,
        scoring=scoring,
    )
    for fold in range(5):
        cv_rows.append(
            {
                "model": model_name,
                "fold": fold + 1,
                "accuracy": scores["test_accuracy"][fold],
                "precision": scores["test_precision"][fold],
                "recall": scores["test_recall"][fold],
                "f1": scores["test_f1"][fold],
            }
        )

cv_fold_results = pd.DataFrame(cv_rows)
cv_summary = (
    cv_fold_results.groupby("model")[["accuracy", "precision", "recall", "f1"]]
    .agg(["mean", "std"])
)

display(cv_summary.round(4))
cv_fold_results.to_csv(OUTPUT_DIR / "five_fold_results.csv", index=False)
cv_summary.to_csv(OUTPUT_DIR / "five_fold_summary.csv")

In [ ]:
cv_plot = cv_fold_results.melt(
    id_vars=["model", "fold"],
    value_vars=["precision", "recall", "f1"],
    var_name="metric",
    value_name="score",
)

plt.figure(figsize=(9, 5))
sns.boxplot(data=cv_plot, x="metric", y="score", hue="model")
sns.stripplot(
    data=cv_plot,
    x="metric",
    y="score",
    hue="model",
    dodge=True,
    color="black",
    alpha=0.5,
    legend=False,
)
plt.title("Five-fold validation: base vs engineered features")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "five_fold_model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

Across five folds, the base model achieved mean precision 0.2256, recall 0.9204, and F1 0.3621. The engineered model achieved mean precision 0.3552, recall 0.9203, and F1 0.5119. Thus, engineered relationships improved precision and F1 without reducing mean recall. Cross-validation does not eliminate dataset limitations, but it shows that the direction of this improvement is not limited to the original holdout split.

## 10. Cost-Sensitivity Analysis

Without real industrial costs, we cannot claim one universal best model. We can still test how the preferred configuration changes as the assumed cost of a missed failure increases relative to a false alarm:

$$Cost = C_{FP}\times FP + C_{FN}\times FN$$

We set $C_{FP}=1$ and vary $C_{FN}$.

In [ ]:
def metric_row(name, actual_values, predicted_values):
    cm = confusion_matrix(actual_values, predicted_values)
    return {
        "configuration": name,
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0]),
        "true_positives": int(cm[1, 1]),
        "precision": precision_score(actual_values, predicted_values, zero_division=0),
        "recall": recall_score(actual_values, predicted_values, zero_division=0),
        "f1": f1_score(actual_values, predicted_values, zero_division=0),
    }


configuration_rows = []
base_probabilities = classification_experiment.predict_proba()
configuration_rows.append(
    metric_row("balanced_threshold_0.5", split_data.y_test, base_probabilities >= 0.5)
)
configuration_rows.append(
    metric_row("balanced_threshold_0.8", split_data.y_test, base_probabilities >= 0.8)
)

weight_five_experiment = FailureClassificationExperiment(class_weight={0: 1, 1: 5})
weight_five_experiment.fit(data)
configuration_rows.append(
    metric_row(
        "failure_weight_5",
        weight_five_experiment.split_data.y_test,
        weight_five_experiment.predict(threshold=0.5),
    )
)

engineered_holdout_model = build_tree_pipeline(engineered_numeric_features)
x_engineered_train = add_engineered_features(split_data.x_train)
x_engineered_test = add_engineered_features(split_data.x_test)
engineered_holdout_model.fit(x_engineered_train, split_data.y_train)
engineered_predictions = engineered_holdout_model.predict(x_engineered_test)
configuration_rows.append(
    metric_row(
        "engineered_threshold_0.5",
        split_data.y_test,
        engineered_predictions,
    )
)

configuration_metrics = pd.DataFrame(configuration_rows)
display(configuration_metrics.round(4))
configuration_metrics.to_csv(OUTPUT_DIR / "cost_configuration_metrics.csv", index=False)

cost_ratios = np.arange(1, 51)
cost_rows = []
for _, row in configuration_metrics.iterrows():
    for missed_failure_cost in cost_ratios:
        cost_rows.append(
            {
                "configuration": row["configuration"],
                "missed_failure_cost_ratio": missed_failure_cost,
                "total_relative_cost": (
                    row["false_positives"]
                    + missed_failure_cost * row["false_negatives"]
                ),
            }
        )

cost_sensitivity = pd.DataFrame(cost_rows)
preferred_by_cost = cost_sensitivity.loc[
    cost_sensitivity.groupby("missed_failure_cost_ratio")["total_relative_cost"].idxmin()
].reset_index(drop=True)

display(preferred_by_cost.iloc[[0, 4, 9, 14, 19, 29, 49]])
cost_sensitivity.to_csv(OUTPUT_DIR / "cost_sensitivity.csv", index=False)
preferred_by_cost.to_csv(OUTPUT_DIR / "preferred_configuration_by_cost.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 5.5))
sns.lineplot(
    data=cost_sensitivity,
    x="missed_failure_cost_ratio",
    y="total_relative_cost",
    hue="configuration",
)
plt.title("Model preference changes with assumed error costs")
plt.xlabel("Cost of one missed failure relative to one false alarm")
plt.ylabel("Total relative cost")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "cost_sensitivity.png", dpi=200, bbox_inches="tight")
plt.show()

The preferred configuration changed at explicit cost assumptions. The failure-weight-5 model had the lowest relative cost for missed-failure ratios 1–5, threshold 0.8 was preferred for ratios 6–14, and the high-recall threshold-0.5 model was preferred from ratio 15 onward. The engineered model improved predictive balance but was not the minimum-cost configuration under these particular assumptions. This is sensitivity analysis, not an economic estimate; real deployment would require costs supplied or validated by maintenance experts.

## 11. Research Conclusions

The integrated results support five conclusions:

1. **Anomaly and failure are not equivalent.** Anomaly-only records have a low observed failure rate, while the classifier detects many failures that are not anomalous.
2. **Agreement is meaningful.** Records flagged by both methods have the highest observed failure rate, 42.86%, although this is not a calibrated probability for future data.
3. **The local false-negative explanation generalizes.** Missed failures as a group show high speed, low torque, and high wear.
4. **Explanations require evaluation.** SHAP additivity tests model faithfulness; repeated LIME runs test local approximation stability.
5. **No model is universally responsible.** Cross-validation uncertainty and explicit error-cost assumptions must accompany model selection.

These findings strengthen the project without claiming production readiness. The AI4I dataset is synthetic, the observed subgroup rates come from one held-out test set, and real maintenance costs and domain validation remain unavailable.

## 12. Saved Evidence

All tables and presentation-ready figures are saved under `outputs/integrated_research_analysis/`. The most useful presentation figures are:

- `signal_agreement_matrix.png`
- `error_group_sensor_means.png`
- `group_shap_heatmaps.png`
- `five_fold_model_comparison.png`
- `cost_sensitivity.png`